<a href="https://colab.research.google.com/github/BraedynL0530/Aenaos/blob/main/PaperProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from transformers import OwlViTForObjectDetection, OwlViTProcessor
from PIL import Image

processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch16")
model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch16")

model.eval()
model.config.output_hidden_states = True


# ----------------------------
# Utils
# ----------------------------

def box_area(box):
    return box[2] * box[3]


def iou(a, b):
    def xyxy(box):
        cx, cy, w, h = box
        return (
            cx - w / 2,
            cy - h / 2,
            cx + w / 2,
            cy + h / 2,
        )

    a1 = xyxy(a)
    a2 = xyxy(b)

    xi1 = max(a1[0], a2[0])
    yi1 = max(a1[1], a2[1])
    xi2 = min(a1[2], a2[2])
    yi2 = min(a1[3], a2[3])

    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)

    area1 = (a1[2] - a1[0]) * (a1[3] - a1[1])
    area2 = (a2[2] - a2[0]) * (a2[3] - a2[1])

    return inter / (area1 + area2 - inter + 1e-6)


# ----------------------------
# NMS-style dedup
# ----------------------------

def dedup(boxes, scores, iou_thresh=0.4):
    keep = []

    idxs = scores.argsort(descending=True)

    for i in idxs:
        box = boxes[i]

        if box_area(box) < 0.005:
            continue

        if all(iou(box, k) < iou_thresh for k in keep):
            keep.append(box)

    return torch.stack(keep) if len(keep) > 0 else torch.empty((0, 4))


# ----------------------------
# Inference
# ----------------------------

image = Image.open("/content/Untitled.jpg")

text_queries = [
    "dog",
    "animal",
    "hand",
    "person",
    "object",
    "foreground object",
]

inputs = processor(images=image, text=text_queries, return_tensors="pt")
outputs = model(**inputs)

boxes = outputs.pred_boxes[0]   # (N, 4)
logits = outputs.logits[0]      # (N, num_queries)

# ----------------------------
# BETTER scoring (ranking not thresholding)
# ----------------------------

scores = logits.max(dim=-1).values  # raw similarity

# take top-k instead of threshold
TOP_K = min(30, scores.shape[0])
scores, idx = scores.topk(TOP_K)

boxes = boxes[idx]

# ----------------------------
# cleanup
# ----------------------------

clean_boxes = dedup(boxes, scores)

# ----------------------------
# world-model-ready output
# ----------------------------

objects = [
    {
        "bbox": b.tolist(),   # cx,cy,w,h
        "state": "unknown"
    }
    for b in clean_boxes
]

print(f"Final objects: {len(objects)}")

for o in objects:
    print(o)

In [ ]:
import torch
import torch.nn as nn

class worldGroundedModel(nn.Module):
  def __init__(self,spatial_dim=128,owl_dim=768,output_dim=256,delta_threshold=0.5):
    super().__init__()
    self.delta_threshold = delta_threshold

    self.spatial_embd = nn.Sequential(
        nn.Linear(4,spatial_dim),
        nn.ReLU(),
        nn.LayerNorm(spatial_dim),
        nn.Linear(spatial_dim,output_dim),
        nn.LayerNorm(output_dim)
    )
    self.features = nn.Sequential(
        nn.Linear(owl_dim,output_dim), # may add a linear inbetween
        nn.ReLU(),
        nn.LayerNorm(output_dim)
    )
    # The input dimension for final_features must be 2 * output_dim after concatenation
    self.final_features = nn.Linear(2 * output_dim, output_dim) # Fixed input_dim based on concatenation


  def forward(self, norm_bbox,norm_quries):
    spatial = self.spatial_embd(norm_bbox)
    visual_features = self.features(norm_quries)
    enriched_features = torch.cat((spatial,visual_features),dim=1)
    final_features = self.final_features(enriched_features)
    return final_features

      def compute_delta(self, old_bbox, old_query, new_bbox, new_query,
                      iou_weight=0.5, feat_weight=0.5):
        """
        Returns a change score in [0,1] and a boolean flag `has_changed`.
        All inputs are single tensors for one entity (no batch dim):
            old_bbox, new_bbox: (4,)   normalized cxcywh
            old_query, new_query: (D,)  raw OWL query embeddings
        """
        # 1. IoU between old and new bounding boxes
        iou = self._box_iou(old_bbox.unsqueeze(0), new_bbox.unsqueeze(0))  # scalar

        # 2. Cosine similarity between query embeddings
        cos_sim = F.cosine_similarity(old_query.unsqueeze(0),
                                       new_query.unsqueeze(0))  # scalar

        # Normalise IoU and similarity to [0,1] (cosine already in [-1,1])
        # For IoU it's already in [0,1]; for cosine we rescale to [0,1]
        cos_norm = (cos_sim + 1.0) / 2.0

        # Combine them (higher = more similar, so change = 1 - similarity)
        similarity = iou_weight * iou + feat_weight * cos_norm
        change_score = 1.0 - similarity

        has_changed = change_score > self.delta_threshold
        return change_score, has_changed

    @staticmethod
    def _box_iou(box1, box2):
        """
        Compute IoU between two boxes in cxcywh format (normalised).
        box1, box2: (1, 4) tensors.
        """
        # Convert to x1y1x2y2
        b1_x1 = box1[..., 0] - box1[..., 2] / 2
        b1_y1 = box1[..., 1] - box1[..., 3] / 2
        b1_x2 = box1[..., 0] + box1[..., 2] / 2
        b1_y2 = box1[..., 1] + box1[..., 3] / 2

        b2_x1 = box2[..., 0] - box2[..., 2] / 2
        b2_y1 = box2[..., 1] - box2[..., 3] / 2
        b2_x2 = box2[..., 0] + box2[..., 2] / 2
        b2_y2 = box2[..., 1] + box2[..., 3] / 2

        inter_x1 = torch.max(b1_x1, b2_x1)
        inter_y1 = torch.max(b1_y1, b2_y1)
        inter_x2 = torch.min(b1_x2, b2_x2)
        inter_y2 = torch.min(b1_y2, b2_y2)

        inter_area = (inter_x2 - inter_x1).clamp(0) * (inter_y2 - inter_y1).clamp(0)
        b1_area = (b1_x2 - b1_x1) * (b1_y2 - b1_y1)
        b2_area = (b2_x2 - b2_x1) * (b2_y2 - b2_y1)
        union = b1_area + b2_area - inter_area + 1e-6
        return inter_area / union

In [ ]:
class linearProjector(nn.Module): #turns bounding box + class into readable inputs for nlp
  def __init__(self,query_dim, bbox_dim, output_dim):
    super().__init__()
    self.bbox_embed = nn.Sequential(
        nn.Linear(bbox_dim,64),
        nn.ReLU(),
        nn.Linear(64,query_dim)
    )
    self.proj = nn.Linear(query_dim, output_dim)

  def forward(self,object_queries,bboxes):
    spatial_emb = self.bbox_embed(bboxes)
    enriched = spatial_emb+object_queries
    tokens = self.proj(enriched)
    return tokens
